# FraudTrace AI - 08: Contradiction Classification Model Training

> **DISCLAIMER: SYNTHETIC DATA ONLY**
> All data used in this notebook is synthetically generated for research, testing, and demonstration purposes. No real police records, personal identification, or banking transactions are included. This system does not make legal guilt determinations; it reconstructs and correlates evidentiary relationships.

### Purpose
Train Logistic Regression and Random Forest models on pairwise forensic features with incident-level splitting.

| Attribute | Description |
| :--- | :--- |
| **Input** | Pairwise feature vectors from synthetic training corpus |
| **Output** | Trained contradiction classification model artifact saved to ml/artifacts/models/ |
| **Target Environment** | Google Colab (CPU / GPU compatible) & Local |
| **Package** | `ml.src` |


In [ ]:
# ====================================================================
# GOOGLE COLAB ENVIRONMENT SETUP
# ====================================================================
import os
import sys

# Clone repository if running inside Google Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('FraudTrace.AI'):
        !git clone https://github.com/Cyberdude441/FraudTrace.AI.git
    %cd FraudTrace.AI
    !pip install -r ml/requirements-colab.txt

# Ensure repository root is on sys.path for local and Colab execution
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"[OK] Environment initialized. Working directory: {os.getcwd()}")


## 1. Incident-Level Data Splitting (Preventing Leakage)
Evidence pairs from the same incident must **never** be split across train and test sets. We split by incident ID to ensure strict generalization.

In [ ]:
from ml.src.data.loader import DataLoader
from ml.src.data.splitter import DatasetSplitter
from ml.src.models.contradiction_model import ContradictionClassifier
import numpy as np

loader = DataLoader(dataset_type="sample")
master_data = loader.load_master_tables()
incidents_df = master_data.get('incidents', None)

splitter = DatasetSplitter(test_size=0.15, val_size=0.15, random_seed=42)
if incidents_df is not None and not incidents_df.empty:
    train_ids, val_ids, test_ids = splitter.split_incidents(incidents_df['incident_id'].tolist())
    print(f"Incident Split -> Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")
else:
    print("Using synthetic split demonstration.")


## 2. Model Training & Comparison
We compare a Logistic Regression baseline against a Random Forest classifier.

In [ ]:
model = ContradictionClassifier(model_type="random_forest")

# Build synthetic training distribution for standard forensic pairwise features
np.random.seed(42)
N = 600
# 14 features: amount_exact, delta_ratio, abs_diff, drift_sec, w15m, w1h, w24h, same_src, indep_src, ent_cnt, ent_ratio, lev, jac, missing
X_corrob = np.random.normal(loc=[1.0, 0.0, 0.0, 120.0, 1.0, 1.0, 1.0, 0.0, 1.0, 2.0, 0.8, 0.9, 0.85, 0.0], scale=0.1, size=(200, 14))
y_corrob = np.zeros(200, dtype=int)  # 0: CORROBORATED

X_conflict = np.random.normal(loc=[0.0, 0.25, 2000.0, 120.0, 1.0, 1.0, 1.0, 0.0, 1.0, 2.0, 0.8, 0.5, 0.4, 0.0], scale=0.1, size=(200, 14))
y_conflict = np.ones(200, dtype=int)  # 1: CONFLICTING

X_drift = np.random.normal(loc=[1.0, 0.0, 0.0, 7200.0, 0.0, 0.0, 1.0, 0.0, 1.0, 2.0, 0.8, 0.8, 0.75, 0.0], scale=0.1, size=(200, 14))
y_drift = np.full(200, 2, dtype=int)  # 2: TIMESTAMP_INCONSISTENCY

X = np.vstack([X_corrob, X_conflict, X_drift])
y = np.concatenate([y_corrob, y_conflict, y_drift])

indices = np.random.permutation(len(X))
X, y = X[indices], y[indices]

train_len = int(0.7 * len(X))
X_train, y_train = X[:train_len], y[:train_len]
X_test, y_test = X[train_len:], y[train_len:]

train_metrics = model.train(X_train, y_train)
print(f"Training Accuracy: {train_metrics['accuracy'] * 100:.2f}%")


## 3. Save Model Artifact
Serialize the trained model to `ml/artifacts/models/contradiction_model_v1.pkl`.

In [ ]:
from ml.src.config import get_artifacts_dir

art_dir = get_artifacts_dir() / "models"
art_dir.mkdir(parents=True, exist_ok=True)
model_path = art_dir / "contradiction_model_v1.pkl"
model.save(str(model_path))
print(f"[OK] Saved contradiction model artifact to {model_path}")
